In [1]:
import pandas as pd
import numpy as np
from plotnine import *
import os

In [214]:
def assign_paabd(dat,train_dat,assign_kinase = True):
    dat["substrate_seq_15aa"] = dat.apply(lambda s: s["substrate_seq"][s["p_ind"]-1-7:s["p_ind"]-1+8],axis = 1)
    dat = dat.loc[dat["substrate_seq_15aa"].apply(lambda s:len(s))==15,:]
    dat.loc[:,"SH2"] = dat["substrate_seq_15aa"].apply(lambda s: s[7]=='Y')
    dat.loc[:,"b14-3-3"] = dat["substrate_seq_15aa"].apply(lambda s: ((s[7]=='S')|(s[7]=='T'))&(s[4]=='R')&(s[9]=='P'))
    dat.loc[:,"WW"] = dat["substrate_seq_15aa"].apply(lambda s: ((s[7]=='S')|(s[7]=='T'))&(s[8]=='P'))
    dat.loc[:,"FHA"] = dat["substrate_seq_15aa"].apply(lambda s: ((s[7]=='S')|(s[7]=='T'))&(s[10] in ["D","I","S","Y"]))
    if assign_kinase:
        dat = pd.merge(dat,train_dat.loc[:,["kinase","kinase_seq"]].drop_duplicates(), on = "kinase_seq")
    return dat 

In [239]:
psites_animal = pd.read_pickle("../kinase_ml/data/230301_all_animal_psites.pkl")
psites_animal = assign_paabd(psites_animal,train_dat = None,assign_kinase = False)
train_dat = pd.read_pickle(os.path.join('../kinase_ml/data',"interactions","interaction_classed_221017.pkl"))
train_dat = assign_paabd(train_dat,train_dat = None,assign_kinase = False)
train_pred_v2 = pd.read_pickle(os.path.join('../kinase_ml/data',"kinbertv2_pred","kinbertv2_DB_Train.pkl"))
train_pred_v1 = pd.read_pickle(os.path.join('../kinase_ml/data',"kinbertv1_pred","kinbertv1_DB_Train.pkl"))
sub1 = psites_animal.loc[:,['substrate','p_ind','p_aa','substrate_seq_15aa']]
sub2 = train_dat.loc[:,['substrate','p_ind','p_aa','substrate_seq_15aa']]
sub12 = pd.concat([sub1,sub2])
train_230621 = pd.read_pickle('../kinase_ml/data/training_data/interations_e2e_1n1p_classed_random_withacc_230621.pkl')
train_230621 = assign_paabd(train_230621,train_dat = None,assign_kinase = False)
sub3 = train_230621.loc[:,['substrate','p_ind','p_aa','substrate_seq_15aa']]
sub123 = pd.concat([sub12,sub3])
DATA_DIR = '../kinase_ml/data'
test_pred_v1 = pd.read_pickle(os.path.join(DATA_DIR,"kinbertv1_pred","kinbertv1_DB_Test.pkl"))
train_pred_v2 = pd.read_pickle(os.path.join(DATA_DIR,"kinbertv2_pred","kinbertv2_DB_Train.pkl"))
train_pred_v1 = pd.read_pickle(os.path.join(DATA_DIR,"kinbertv1_pred","kinbertv1_DB_Train.pkl"))
test_pred_v2 = pd.read_pickle(os.path.join(DATA_DIR,"kinbertv2_pred","kinbertv2_DB_Test.pkl"))
test_pred_v1 = assign_15aa(test_pred_v1)
train_pred_v2 = assign_15aa(train_pred_v2)
train_pred_v1 = assign_15aa(train_pred_v1)
test_pred_v2 = assign_15aa(test_pred_v2)
test_train_v12 = pd.concat([test_pred_v1,train_pred_v2,train_pred_v1,test_pred_v2])
print("number of substrates mapped ")
print(specificity_df.loc[specificity_df['substrate'].isin(test_train_v12['substrate_seq_15aa']),['substrate']].drop_duplicates().shape[0])

/tmp/ipykernel_21315/3047277841.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_21315/3047277841.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_21315/3047277841.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_21315/3

In [469]:
sty_spike_ins.loc[~sty_spike_ins['substrate_seq_15aa'].isin(sub12345['substrate_seq_15aa']),:]

,Unnamed: 0,substrate_seq_15aa,kinase,logit,PAABD,p_aa,set
678,59492,VGAPIAVSAFYAESC,P53350,-4.660689,FHA,S,train_tn
698,59492,VGAPIAVSAFYAESC,P53350,-4.660689,FHA,T,train_tn


In [ ]:
sub12345 = pd.concat([sub1234,sty_background])

In [470]:
sub12345.to_pickle('substrates_background_12345.pkl')

In [352]:
st_sub_all = pd.read_csv('../kinase_ml/code/wetlab_planning/st_sub_all.csv')
y_sub_all = pd.read_csv('../kinase_ml/code/wetlab_planning/y_sub_all.csv')
sty_sub_all = pd.concat([st_sub_all,y_sub_all])

In [423]:
substrate_aa_acc_map = pd.merge(sty_sub_all,sub12345,on = 'substrate_seq_15aa').drop_duplicates()

In [436]:
n = sty_sub_all['substrate_seq_15aa'].drop_duplicates().shape[0]
print(f'we put in total {n} substrates into the screening')
n = sty_sub_all.loc[sty_sub_all['substrate_seq_15aa'].isin(sub12345['substrate_seq_15aa']),:].drop_duplicates('substrate_seq_15aa').shape[0]
print(f'{n} substrates could be mapped back with uniprot acc and residue')
n = sty_sub_all.loc[~sty_sub_all['substrate_seq_15aa'].isin(sub12345['substrate_seq_15aa']),:].shape[0]
print(f'{n} substrates could not be mapped back with uniprot acc and residue')

we put in total 3918 substrates into the screening
3917 substrates could be mapped back with uniprot acc and residue
1 substrates could not be mapped back with uniprot acc and residue


In [437]:
print('in the actual screening there are more than what we put in due to mutations')

in the actual screening there are more than what we put in due to mutations


In [ ]:
dat_all_blib = pd.read_csv("./dat_all_blib_2batch.csv")
screen_df = dat_all_blib.loc[:,['barcode_seq','pre_1','mtx_1','pre_2','mtx_2','kinase_name','substrate_aa','paabd']]
screen_df.to_pickle('./decoupling_out/raw_screen_count.pkl')

In [2]:
count_df = pd.read_pickle('./decoupling_out/raw_screen_count.pkl')

In [453]:
n = count_df['substrate_aa'].drop_duplicates().shape[0]
print(f'there are in total {n} substrates in the actual screening')
n = count_df.loc[count_df['substrate_aa'].isin(sub12345['substrate_seq_15aa']),:].drop_duplicates('substrate_aa').shape[0]
print(f'{n} substrates could be mapped back with uniprot acc and residue')
n = count_df.loc[~count_df['substrate_aa'].isin(sub12345['substrate_seq_15aa']),:].drop_duplicates('substrate_aa').shape[0]
print(f'{n} substrates could not be mapped back with uniprot acc and residue')

there are in total 5335 substrates in the actual screening
3888 substrates could be mapped back with uniprot acc and residue
1447 substrates could not be mapped back with uniprot acc and residue


In [465]:
print('number of kinase distribution linked to substrates that are not mapped to a known acc')
count_df.loc[~count_df['substrate_aa'].isin(sub12345['substrate_seq_15aa']),:].drop_duplicates(['substrate_aa','kinase_name']).groupby('substrate_aa').count().reset_index().groupby('kinase_name').count().iloc[:,[0]]

number of kinase distribution linked to substrates that are not mapped to a known acc


,substrate_aa
kinase_name,
1,1436
2,8
6,1
13,1
14,1


In [466]:
print('number of kinase distribution linked to substrates that are mapped to a known acc')
count_df.loc[count_df['substrate_aa'].isin(sub12345['substrate_seq_15aa']),:].drop_duplicates(['substrate_aa','kinase_name']).groupby('substrate_aa').count().reset_index().groupby('kinase_name').count().iloc[:,[0]]

number of kinase distribution linked to substrates that are mapped to a known acc


,substrate_aa
kinase_name,
1,29
2,26
3,27
4,39
5,109
6,459
7,823
8,603
9,87


In [467]:
print('given that most of the unmapped substrates exists only in only one kinase,it is most likely that these substrates should be removed prior to the inference')

given that most of the unmapped substrates exists only in only one kinase,it is most likely that these substrates should be removed prior to the inference


In [473]:
count_df_mapped = count_df.loc[count_df['substrate_aa'].isin(sub12345['substrate_seq_15aa']),:]

In [474]:
count_df_mapped.to_pickle('./decoupling_out/raw_screen_count_mapped.pkl')

In [475]:
count_df_mapped.to_csv('./decoupling_out/raw_screen_count_mapped.csv')

In [476]:
count_df_mapped.shape

(262972, 8)

In [477]:
count_df.shape

(265601, 8)

In [6]:
count_df = pd.read_pickle('./decoupling_out/raw_screen_count.pkl')
count_df['lib'] = 'ST'
count_df.loc[count_df['paabd']=='sh2','lib'] = 'Y'

In [8]:
count_df = count_df.drop_duplicates(['kinase_name','substrate_aa'])

In [10]:
count_df.groupby('lib').count()

,barcode_seq,pre_1,mtx_1,pre_2,mtx_2,kinase_name,substrate_aa,paabd
lib,,,,,,,,
ST,23452,23452,23452,23452,23452,23452,23452,23452
Y,13943,13943,13943,13943,13943,13943,13943,13943
